# 02 — Feature Engineering
Reads the raw `.parquet` files produced by `01_fetch_training_data.ipynb` and:
1. Computes all **11 normalised features** (matching `state_builder.py` exactly).
2. Builds **sliding window sequences** of length `LOOKBACK_BARS=60`.
3. Performs a **time-ordered 80/20 train/val split** (no shuffle — avoids lookahead bias).
4. Saves each ticker's arrays as `{TICKER}_train.npz` and `{TICKER}_val.npz`.

**Input:**  `/content/drive/MyDrive/algo_trader/data/raw/{TICKER}.parquet`  
**Output:** `/content/drive/MyDrive/algo_trader/data/features/{TICKER}_{split}.npz`

In [ ]:
!pip install -q pyarrow pandas numpy pytz tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys
RAW_DIR  = '/content/drive/MyDrive/algo_trader/data/raw'
FEAT_DIR = '/content/drive/MyDrive/algo_trader/data/features'
os.makedirs(FEAT_DIR, exist_ok=True)
print('Directories ready ✓')

In [ ]:
# Clone the repo (or adjust to your Drive path if already cloned)
REPO_URL = 'https://github.com/rohanpatrick568/deepscalper_copilot.git'
REPO_DIR = '/content/deepscalper_copilot'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

# Pointing to the 'colab' directory allows us to import the 'deepscalper' package inside it
import sys
COLAB_DIR = os.path.join(REPO_DIR, 'algo_trader', 'colab')
if COLAB_DIR not in sys.path:
    sys.path.insert(0, COLAB_DIR)

print('Repo path updated for deepscalper package ✓')

In [ ]:
SP100_TICKERS = [
    'AAPL','MSFT','AMZN','NVDA','GOOGL','GOOG','META','TSLA','BRK.B','UNH',
    'LLY','JPM','V','AVGO','XOM','MA','COST','PG','JNJ','HD',
    'ABBV','ORCL','BAC','WMT','NFLX','KO','CRM','CVX','MRK','AMD',
    'CSCO','PEP','ACN','LIN','TMO','MCD','ABT','IBM','GE','TXN',
    'PM','GS','ISRG','CAT','AXP','SPGI','AMGN','RTX','PFE','BKNG',
    'DHR','MS','INTU','BLK','T','VRTX','HON','NEE','UNP','SYK',
    'C','LOW','TJX','ADP','GILD','DE','PANW','BMY','AMAT','MDT',
    'PLD','SBUX','ADI','TMUS','ETN','SCHW','CB','MMC','BA','SO',
    'MO','WFC','UPS','CI','MDLZ','DUK','CL','INTC','REGN','PH',
    'EOG','SLB','ELV','APD','MCK','COF','ZTS','BSX','GEV','CME',
]
print(f'Processing {len(SP100_TICKERS)} tickers')

In [ ]:
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

from deepscalper.utils import compute_macro_features, compute_micro_features, compute_day_starts

summary = []
skipped = []

for ticker in tqdm(SP100_TICKERS, desc='Feature engineering'):
    raw_path = f'{RAW_DIR}/{ticker}.parquet'
    out_path = f'{FEAT_DIR}/{ticker}.npz'

    if os.path.exists(out_path):
        print(f'{ticker}: already processed — skipping.')
        skipped.append(ticker)
        continue

    if not os.path.exists(raw_path):
        print(f'WARNING: {raw_path} not found — skipping {ticker}.')
        skipped.append(ticker)
        continue

    bars = pd.read_parquet(raw_path)
    bars.columns = [c.lower() for c in bars.columns]
    bars = bars[['open', 'high', 'low', 'close', 'volume']].astype(float)

    macro_feats = compute_macro_features(bars)   # (n_bars, 11)
    lob_feats   = compute_micro_features(bars)   # (n_bars, 5)
    close_arr   = bars['close'].to_numpy(dtype=np.float32)
    day_starts  = np.array(compute_day_starts(bars.index), dtype=np.int32)

    if len(macro_feats) < 100:
        print(f'WARNING: {ticker} has only {len(macro_feats)} bars — skipping.')
        skipped.append(ticker)
        continue

    np.savez_compressed(
        out_path,
        macro_feats  = macro_feats.astype(np.float32),
        lob_feats    = lob_feats.astype(np.float32),
        close_prices = close_arr,
        day_starts   = day_starts,
    )

    summary.append({
        'ticker' : ticker,
        'n_bars' : len(macro_feats),
        'n_days' : len(day_starts),
    })

print('\n=== FEATURE ENGINEERING SUMMARY ===')
if summary:
    print(pd.DataFrame(summary).to_string(index=False))
if skipped:
    print(f'\nSkipped: {skipped}')
print('\nDone ✅')